# Dos modelos finales y evaluación en test

Los modelos se entrenan desde cero con train/val y predicen el test 2024/25–2025/26 sin ningún ajuste adicional.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Entrenamiento final y evaluación cerrada

Las decisiones se leen del archivo congelado. Si la misma ejecución ya fue evaluada, se leen sus resultados en lugar de volver a seleccionar.

In [2]:
from mlcore.experiment import evaluate
final=evaluate(OUT)
display(final)
display(pd.read_csv(OUT/'primary_comparisons.csv'))

,split,candidate,n,log_loss,RPS,Brier,AUC_macro,accuracy,balanced_accuracy,ECE_masa,ECE_ancho,acierto_ganador_condicional,recall_empate
0,validacion,prior,18184,1.071974,0.230953,0.648671,0.496274,0.437747,0.333333,0.013381,0.009633,0.582510,0.000000
1,validacion,consenso,18184,0.966435,0.195253,0.573667,0.684047,0.538440,0.463848,0.014312,0.012621,0.716575,0.000885
2,validacion,Pinnacle_respaldo,18184,0.965690,0.195084,0.573224,0.684675,0.538220,0.463744,0.013850,0.011183,0.716356,0.001328
3,validacion,modelo_con,18184,0.965199,0.194988,0.572867,0.684391,0.538715,0.464498,0.008538,0.006877,0.716795,0.001992
4,validacion,modelo_sin,18184,0.977599,0.198801,0.581351,0.671613,0.531401,0.454404,0.015104,0.013440,0.707135,0.000000
5,test_revisado,prior,5999,1.074963,0.229844,0.650572,0.500000,0.433072,0.333333,0.008585,0.008585,0.582642,0.000000
6,test_revisado,consenso,5999,0.966804,0.193444,0.574675,0.683804,0.538090,0.468773,0.022090,0.016843,0.723705,0.000649
7,test_revisado,Pinnacle_respaldo,5999,0.966263,0.193311,0.574416,0.683743,0.538090,0.468899,0.020755,0.015815,0.723929,0.001948
8,test_revisado,modelo_con,5999,0.965034,0.193014,0.573756,0.683438,0.538590,0.470351,0.016186,0.011806,0.724153,0.001948
9,test_revisado,modelo_sin,5999,0.979290,0.197538,0.583586,0.667776,0.523587,0.453306,0.012118,0.012512,0.704418,0.000000


,split,a,b,delta,lo,hi,p,temporadas_mejora,p_BH
0,validacion,modelo_con,consenso,-0.001236,-0.001955,-0.000500,0.002000,5/6,0.002500
1,validacion,modelo_con,Pinnacle_respaldo,-0.000491,-0.001085,0.000137,0.120776,5/6,0.120776
2,validacion,modelo_sin,consenso,0.011165,0.009075,0.013296,0.000400,0/6,0.000667
3,validacion,modelo_sin,Pinnacle_respaldo,0.011910,0.009770,0.014041,0.000400,0/6,0.000667
4,validacion,modelo_sin,modelo_con,0.012400,0.010266,0.014566,0.000400,0/6,0.000667
5,test_revisado,modelo_con,consenso,-0.001769,-0.002877,-0.000682,0.000800,2/2,0.001000
6,test_revisado,modelo_con,Pinnacle_respaldo,-0.001229,-0.002187,-0.000282,0.013197,2/2,0.013197
7,test_revisado,modelo_sin,consenso,0.012486,0.008683,0.016580,0.000400,0/2,0.000667
8,test_revisado,modelo_sin,Pinnacle_respaldo,0.013027,0.009245,0.017174,0.000400,0/2,0.000667
9,test_revisado,modelo_sin,modelo_con,0.014256,0.010203,0.018530,0.000400,0/2,0.000667


**Qué encontramos / decisión.** Ambos enfoques reciben evaluación sobre los mismos partidos. La identidad del test no se restaura borrando un modelo.

## Probabilidades, cuotas justas y backtest

La semejanza al mercado se calcula por referencia. La rentabilidad utiliza tau fijo y solo precios considerados fiables.

In [3]:
display(pd.read_csv(OUT/'similarity.csv'))
display(pd.read_csv(OUT/'backtest.csv'))
import joblib
df,sets=read_data(OUT); te=df[df.Season.isin(TEST)]
for branch in ['con','sin']:
    model=joblib.load(OUT/f'modelo_{branch}_cuotas.joblib'); p=model.predict_proba(te.iloc[:3])
    print(branch,'probabilidades',p,'cuotas justas',1/p)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1)

,split,reference,corr_logit,corr_pD,mismo_favorito,fraccion_ganancia_prior
0,validacion,consenso,0.947933,0.867158,0.908546,0.894212
1,validacion,Pinnacle_respaldo,0.947345,0.860329,0.907996,0.887944
2,test_revisado,consenso,0.948776,0.864334,0.908651,0.884556
3,test_revisado,Pinnacle_respaldo,0.948232,0.864036,0.908485,0.880157


,split,candidate,casa,tau,apuestas,ROI,lo,hi,EV_prometido
0,validacion,prior,B365,0.02,17159,-0.110958,-0.137723,-0.082515,0.891431
1,validacion,prior,PS,0.02,17798,-0.083434,-0.109931,-0.055974,0.947392
2,validacion,consenso,B365,0.02,809,0.253684,0.061971,0.458382,0.048979
3,validacion,consenso,PS,0.02,2039,-0.126719,-0.248057,0.001700,0.052877
4,validacion,Pinnacle_respaldo,B365,0.02,1509,0.074274,-0.053890,0.206302,0.056842
5,validacion,Pinnacle_respaldo,PS,0.02,0,NaN,NaN,NaN,NaN
6,validacion,modelo_con,B365,0.02,2578,0.064996,-0.016364,0.149800,0.057717
7,validacion,modelo_con,PS,0.02,3881,-0.041667,-0.103668,0.021341,0.042203
8,validacion,modelo_sin,B365,0.02,12298,-0.043657,-0.075548,-0.011129,0.160741
9,validacion,modelo_sin,PS,0.02,14908,-0.031011,-0.058611,-0.001187,0.168362


/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


con probabilidades [[0.70788611 0.18287659 0.1092373 ]
 [0.63610274 0.21746916 0.1464281 ]
 [0.33036703 0.29360995 0.37602303]] cuotas justas [[1.41265663 5.46816831 9.15438226]
 [1.57207308 4.59835319 6.82929038]
 [3.02693645 3.40587917 2.65941161]]


sin probabilidades [[0.58840915 0.24976659 0.16182425]
 [0.62040771 0.23873049 0.1408618 ]
 [0.30756647 0.28560196 0.40683158]] cuotas justas [[1.69949769 4.00373799 6.1795435 ]
 [1.61184329 4.18882401 7.0991566 ]
 [3.25132973 3.50137657 2.45801963]]


**Qué encontramos / decisión.** Las cuotas justas no incluyen margen comercial. Para usar el modelo deportivo se proporcionan sus variables históricas, nunca el resultado del partido.

## Entregables desde la misma tabla

Las figuras, el informe y las 13 diapositivas se generan desde esta ejecución, no desde cifras copiadas a mano.

In [4]:
from mlcore.report import render
render(OUT)
print('Fuentes de informe y presentacion regeneradas; compilar PDF con scripts/report/build.py')

/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


Figuras, informe y 13 diapositivas actualizados desde 01afe756130bafc6
Fuentes de informe y presentacion regeneradas; compilar PDF con scripts/report/build.py


**Qué encontramos / decisión.** Las conclusiones de ambas ramas deben seguir los resultados corregidos. La siguiente evaluación independiente requiere partidos futuros.